# Expanded Practice Exercise (R version): A/B Testing Website Versions
## Two-Sample t-Test with Assumption Checks, Effect Size, CI, More Practice & Simulation

**Goal:** Practice the full workflow of a two-sample t-test in **R** while considering your audience (data literacy, subject knowledge) when interpreting and reporting results.

This is the **SKELETON R version** — follow the markdown instructions and fill in the code cells. Use the `solution_ab_test_ttest_R.ipynb` to check your work or see alternate R approaches after attempting.

### Scenario
Same as Python version: Company A/B tested old vs new website. 50 visitors each. Time spent recorded in `version_time.csv`.
You will perform a complete professional analysis suitable for a data analysis report, now entirely in R.

**Packages you will need (run once):** `install.packages(c("tidyverse", "ggplot2"))`  (tidyverse includes dplyr, readr, etc.)


## Flowchart of the Desired Analysis Outcome

```mermaid
flowchart TD
    Start[Start] --> Load[Load Data &amp; EDA<br/>Histograms, Summary Stats, Groupby]
    Load --> Hypotheses[Formulate Hypotheses<br/>H0: μ_new = μ_old<br/>Ha: μ_new ≠ μ_old | α = 0.05 two-sided]
    Hypotheses --> Assumptions{Check Assumptions<br/>1. Normality (Shapiro-Wilk / QQ-plot / Hist)<br/>2. Equal Variance (Levene / var.test)}
    Assumptions -->|Pass| TTest[Run Two-Sample t-test<br/>t.test(var.equal = TRUE)<br/>+ Alternates: wilcox.test, boot, manual]
    Assumptions -->|Fail or Borderline| NonParam[Consider Wilcoxon / Mann-Whitney<br/>or transform data / CLT justification]
    TTest --> EffectSize[Compute Effect Size<br/>Cohen's d + Interpretation]
    TTest --> CI[Compute 95% CI for Mean Difference<br/>(t.test gives it directly)]
    EffectSize --> Interpret[Interpret Results<br/>p-value vs α<br/>Statistical + Practical Significance]
    CI --> Interpret
    NonParam --> Interpret
    Interpret --> Audience[Consider Audience for Reporting<br/>- Executives: High-level business impact<br/>- Data team: Full stats, assumptions, code<br/>- Non-technical: Simple language + viz]
    Audience --> Conclusion[Conclusion &amp; Recommendations<br/>Rollout decision? Next experiments?]
    Conclusion --> Sim[Simulation: Power Analysis<br/>Modify params → observe power / Type I error]
    Sim --> End[End: Practice + Document Insights]
```

**Note:** Mermaid flowchart renders in JupyterLab, VS Code, nbviewer, or paste at https://mermaid.live. It shows the logical flow of a complete, audience-aware A/B test analysis (language agnostic).


## 1. Setup, Libraries and Data Loading

**Instructions:**
- Load libraries: `tidyverse` (or at minimum `readr`, `dplyr`, `ggplot2`)
- Read the CSV into a tibble/data.frame called `data`
- Create two vectors: `old` and `new` (or work with filtered data frames)
- Print structure, first rows, and counts per version


In [ ]:
# TODO: Install if needed (run once)
# install.packages(c("tidyverse", "ggplot2"))

library(tidyverse)
library(ggplot2)

# TODO: Load data
data <- read_csv("version_time.csv")

# TODO: Inspect
print(dim(data))
print(table(data$version))
print(head(data))

# TODO: Create vectors for each group (or use filter + pull)
old <- data$time_minutes[data$version == "old"]
new <- data$time_minutes[data$version == "new"]

print(length(old))
print(length(new))


## 2. Exploratory Data Analysis (EDA) & Visualization

**Instructions:**
1. Compute summary statistics for each group (mean, sd, min, max, n). Use `summary()`, `mean()`, `sd()`, or `group_by` + `summarise`.
2. Create an overlaid histogram with `ggplot2` (use `geom_histogram` + `alpha` for transparency, `fill` by version).
3. (Bonus) Create side-by-side boxplots with `ggplot2`.
4. Note from visuals: Does new version appear to increase time? Any obvious skewness or outliers?

**Audience tip:** For less data-literate stakeholders, clear histograms + mean comparison with plain labels are safest.


In [ ]:
# TODO: Summary statistics
print("=== OLD version ===")
print(summary(old))
print(sd(old))

print("=== NEW version ===")
print(summary(new))
print(sd(new))

# TODO: Overlaid histogram with ggplot2
ggplot(data, aes(x = time_minutes, fill = version)) +
  geom_histogram(alpha = 0.6, position = "identity", bins = 15, color = "white") +
  labs(title = "Distribution of Time Spent on Website by Version",
       x = "Time spent (minutes)", y = "Count") +
  theme_minimal() +
  scale_fill_manual(values = c("old" = "steelblue", "new" = "coral"))

# TODO (optional): Boxplots
# ggplot(data, aes(x = version, y = time_minutes, fill = version)) +
#   geom_boxplot(alpha = 0.7) +
#   labs(title = "Boxplot Comparison of Time by Version") +
#   theme_minimal()


## 3. Formulate Hypotheses and Choose Significance Level

**Instructions:**
- Clearly state H₀ and Hₐ in a markdown cell.
- Choose α = 0.05 and justify two-sided test.

Example:
- H₀: μ_new = μ_old (no difference in mean time spent)
- Hₐ: μ_new ≠ μ_old


In [ ]:
# No executable code needed here — write your hypotheses in the markdown cell above.
cat("Hypotheses stated above. α = 0.05 (two-sided test)\n")


## 4. Check Statistical Assumptions

**Instructions:**
1. Create Q-Q plots for both groups (use `qqnorm()` + `qqline()` or `ggplot2` with `stat_qq()`).
2. Run Shapiro-Wilk test: `shapiro.test(old)` and `shapiro.test(new)`. Print p-values.
3. Check equal variances: `var.test(old, new)` (F-test) or install `car` and use `leveneTest()` for Levene’s test.
4. In a markdown cell below: Decide if assumptions are reasonably met for a parametric t-test. What if they are violated?


In [ ]:
# TODO: Q-Q plots (base R)
par(mfrow = c(1, 2))
qqnorm(old, main = "Q-Q Plot: Old Version"); qqline(old, col = "red")
qqnorm(new, main = "Q-Q Plot: New Version"); qqline(new, col = "red")
par(mfrow = c(1, 1))

# TODO: Shapiro-Wilk tests
shap_old <- shapiro.test(old)
shap_new <- shapiro.test(new)
print(shap_old)
print(shap_new)

# TODO: Variance test (F-test) or Levene
var_test <- var.test(old, new)
print(var_test)

# For full Levene's test (more robust):
# install.packages("car")
# library(car)
# leveneTest(time_minutes ~ version, data = data)

# TODO: Write your decision in a markdown cell below this one


## 5. Perform the Two-Sample t-Test

**Instructions:**
1. Use `t.test(new, old, var.equal = TRUE)` (or `var.equal = FALSE` for Welch).
2. Print the full result object — it includes t-statistic, df, p-value, 95% CI, and means.
3. Extract and print p-value and decide significance at α = 0.05.
4. Also print the observed mean difference.

**Alternate approaches (see solution):**
- `wilcox.test()` (non-parametric Mann-Whitney / Wilcoxon rank sum)
- Bootstrap CI using `replicate()` + `sample()`


In [ ]:
# TODO: Run two-sample t-test (equal variance assumed)
t_result <- t.test(new, old, var.equal = TRUE)
print(t_result)

# TODO: Extract key values
pval <- t_result$p.value
alpha <- 0.05
significant <- pval < alpha
cat(sprintf("p-value = %.6f\n", pval))
cat(sprintf("Significant at α=%.2f? %s\n", alpha, significant))

mean_diff <- mean(new) - mean(old)
cat(sprintf("Observed mean difference (new - old) = %.3f minutes\n", mean_diff))


## 6. Effect Size and Confidence Interval

**Instructions:**
1. Calculate Cohen’s d manually (pooled SD version):
   d = (mean_new - mean_old) / sqrt( ((n_old-1)*var_old + (n_new-1)*var_new) / (n_old + n_new - 2) )
2. Interpret: |d| ≈ 0.2 small, 0.5 medium, 0.8 large.
3. The `t.test()` object already contains the 95% CI — print `t_result$conf.int`.

Print everything and interpret practical significance in a markdown cell.


In [ ]:
# TODO: Cohen's d (manual calculation)
n_old <- length(old)
n_new <- length(new)
var_old <- var(old)
var_new <- var(new)
pooled_var <- ((n_old - 1) * var_old + (n_new - 1) * var_new) / (n_old + n_new - 2)
pooled_sd <- sqrt(pooled_var)
cohens_d <- mean_diff / pooled_sd
cat(sprintf("Cohen's d = %.3f  (medium effect around 0.5)\n", cohens_d))

# TODO: 95% CI (already in t.test result)
cat("95% CI for mean difference:\n")
print(t_result$conf.int)


## 7. More Practice Exercises (R)

Complete these in new code cells. Check answers in the solution notebook.

**Practice 1:** Re-run significance decision with α = 0.01 and α = 0.10.

**Practice 2:** One-sided test (`alternative = "greater"` in t.test). Interpret in business context.

**Practice 3:** Compute & print a 90% CI (use `conf.level = 0.90` in t.test).

**Practice 4:** Run `wilcox.test(new, old, alternative = "two.sided")`. Compare conclusion to t-test.

**Practice 5 (Advanced):** Bootstrap 95% CI for mean difference using `replicate(5000, ...)` + `sample(..., replace = TRUE)`.


## 8. Simulation Section: Statistical Power & Sensitivity Analysis (R)

**Goal:** Modify parameters and instantly see how power or Type I error rate changes.

**Instructions:**
1. Change the values of `true_mean_new`, `n_per_group`, `sigma`, or `n_simulations`.
2. Re-run the cell.
3. Observe power (when true effect exists) or Type I error rate (when true_mean_new == true_mean_old).

Use `replicate()` for clean vectorized simulation in R.


In [ ]:
set.seed(42)

# === MODIFIABLE PARAMETERS - CHANGE THESE ===
true_mean_old <- 23.53
true_mean_new <- 26.88   # change to 23.53 for null (expect ~alpha)
sigma <- 5.3
n_per_group <- 50
n_simulations <- 1000
alpha <- 0.05

# === Simulation using replicate (R-idiomatic) ===
sim_pvals <- replicate(n_simulations, {
  old_sim <- rnorm(n_per_group, true_mean_old, sigma)
  new_sim <- rnorm(n_per_group, true_mean_new, sigma)
  t.test(new_sim, old_sim, var.equal = TRUE)$p.value
})

significant_count <- sum(sim_pvals < alpha)
power_estimate <- significant_count / n_simulations

label <- if (abs(true_mean_new - true_mean_old) > 0.1) "Estimated Power" else "Estimated Type I Error Rate"
cat(sprintf("%s: %.3f (based on %d simulations)\n", label, power_estimate, n_simulations))
cat(sprintf("Mean p-value across simulations: %.4f\n", mean(sim_pvals)))

# TODO: Histogram of p-values
hist(sim_pvals, breaks = 30, col = "purple", main = paste("p-value Distribution from", n_simulations, "Simulated Experiments"),
     xlab = "p-value", ylab = "Frequency")
abline(v = alpha, col = "red", lwd = 2, lty = 2)
legend("topright", legend = paste("α =", alpha), col = "red", lty = 2, lwd = 2)

cat("\nTip: Under null, p-values ~ Uniform[0,1]. Peak near 0 = good power to detect the simulated effect.\n")


## 9. Conclusion & Audience-Aware Reporting (Your Turn in R)

After finishing the analysis, write a short Conclusion following data analysis report structure.
Tailor the language to different audiences (executives vs data team vs non-technical), exactly as in the Python version.

Use the same prompt structure provided in the Python skeleton.
